In [2]:
import os 
import pandas as pd 
import numpy as np 
from pathlib import Path
from dataclasses import dataclass

In [3]:
pwd 

'c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\Research'

In [4]:
@dataclass
class DataGamificaitionConfig:
    root_dir : Path
    data_path : str 
    local_data_file : str 
    incident_points : str
    analyst_points  : str
    badgeresults : Path
    badgesummary : Path
    levelresults : Path
    levelsummary : Path
    leaderboardfile : Path
    topfile : Path  
    bottomfile : Path
    teamfile : Path 
    groupfile : Path 
    achievementfile : Path
    summaryfile : Path

In [5]:
os.chdir('c:\\Users\\RSR\\PYTHON\\ServicenowGamification\\src')

In [6]:
from ServicenowGamification.Utils.Common import *
from ServicenowGamification.Constant import * 
from ServicenowGamification.Entity.Config_Entity import *

In [7]:
os.chdir(r"C:\Users\RSR\PYTHON\ServicenowGamification")

In [8]:
class ConfigurationManager:
    def __init__(self,
                 CONFIG_PATH=CONFIG_FILE_PATH,
                 PARAMS_PATH=PARAMS_FILE_PATH
                 ):
         print(CONFIG_PATH)
                 
         print(f"path config yaml {Path(CONFIG_PATH)}")
         self.config=Read_Yaml(Path(CONFIG_PATH))
         self.params=Read_Yaml(Path(PARAMS_PATH))
         self.Data_Gamification = self.config.Data_Gamification
    def get_datagamificationconfig(self)->DataGamificationConfig:
         Config=self.Data_Gamification
         create_directories([Config.root_dir])
         data_gamification_config=DataGamificationConfig(
              root_dir =Config.root_dir,
              data_path=Config.data_path,
              local_data_file=Config.local_data_file,
              incident_points=Config.incident_points,
              analyst_points=Config.analyst_points,
              badgeresults=Config.badgeresults,
              badgesummary=Config.badgesummary,
              levelresults=Config.levelresults,
              levelsummary= Config.levelsummary,
              leaderboardfile= Config.leaderboardfile, 
              topfile= Config.topfile, 
              bottomfile= Config.bottomfile,
              teamfile= Config.teamfile,
              groupfile= Config.groupfile ,
              achievementfile= Config.achievementfile ,
              summaryfile= Config.summaryfile              
         )
         return data_gamification_config




In [ ]:
class Gamification: 
    def __init__(self,config:DataGamificaitionConfig):
        self.config=config
        self.params = Read_Yaml(PARAMS_FILE_PATH)
        
    ##################################################################
    # Calculate Points
    ##################################################################
        

    def calculate_points(self):
        print("##################################################################")
        print(f" # Point calculation")
        print("##################################################################")
        df = pd.read_csv(self.config.data_path)
        total_points = []

        for _, row in df.iterrows():

            points = 0

            ##########################################################
            # Incident Closed
            ##########################################################

            if str(row["state"]).lower() == "Closed":

                points += self.params.gamification.points["incident_closed"]

            ##########################################################
            # SLA Met
            ##########################################################

            if row["sla_status"] == "Met":

                points += self.params.gamification.points["sla_met"]

            ##########################################################
            # First Call Resolution
            ##########################################################

            if row["fcr"] == 1:

                points += self.params.gamification.points["first_call_resolution"]

            ##########################################################
            # High Priority Bonus
            ##########################################################

            if str(row["priority"]) == "1 - Critical":

                points += self.params.gamification.points["high_priority_bonus"]

            ##########################################################
            # Customer Rating Bonus
            ##########################################################

            if "customer_rating" in df.columns:

                if row["customer_rating"] >= 4.5:

                    points += self.params.gamification.points["customer_rating_bonus"]

            ##########################################################
            # Reopen Penalty
            ##########################################################

            if "reopen_count" in df.columns:

                if row["reopen_count"] > 0:

                    points += abs(self.params.gamification.points["reopen_penalty"]) * row["reopen_count"] * -1

            ##########################################################
            # Reassignment Penalty
            ##########################################################

            if "reassignment_count" in df.columns:

                if row["reassignment_count"] > 0:

                    points += abs(self.params.gamification.points["reassignment_penalty"]) * row["reassignment_count"] * -1

            total_points.append(points)

        df["points"] = total_points

        return df
    ##################################################################
    # Analyst Wise Summary
    ##################################################################

    def analyst_summary(self,df):
            df=df
            summary = (

                df

                .groupby("assigned_to")

                .agg(

                    total_incidents=("number", "count"),

                    total_points=("points", "sum"),

                    sla_met=("sla_status", lambda x: (x == "Met").sum()),

                    average_resolution=("resolution_hours", "mean")

                )

                .reset_index()

            )

            summary["average_resolution"] = summary[
                "average_resolution"
            ].round(2)

            summary = summary.sort_values(

                by="total_points",

                ascending=False

            )

            return summary
        ##################################################################
    # Save Results
    ##################################################################

    def p_save(self,df):
         
        incident_path =self.config.incident_points
        
        df.to_csv(

            incident_path,

            index=False

        )

        summary_path =self.config.analyst_points

        df_analyst_summary = self.analyst_summary(df)

        df_analyst_summary.to_csv(

            summary_path,

            index=False

        )

        print(f"Incident points saved to : {incident_path}")

        print(f"Analyst summary saved to : {summary_path}")

        return df,df_analyst_summary
           

    def assign_badges(self,df):
        print("##################################################################")
        print(f" #  Badge Assignment")
        print("##################################################################") 
        badge_list = []

        for _, row in df.iterrows():

            points = row["points"]

            if points >= self.params.gamification.badges["legend"]["min_points"]:

                badge = "Legend"

            elif points >= self.params.gamification.badges["platinum"]["min_points"]:

                badge = "Platinum"

            elif points >= self.params.gamification.badges["gold"]["min_points"]:

                badge = "Gold"

            elif points >= self.params.gamification.badges["silver"]["min_points"]:

                badge = "Silver"

            elif points >= self.params.gamification.badges["bronze"]["min_points"]:

                badge = "Bronze"

            else:

                badge = "Rookie"

            badge_list.append(badge)

        df["badge"] = badge_list

        return df 

    ##################################################################
    # Badge Summary
    ##################################################################

    def badge_summary(self,df):
        df=df

        summary = (

            df

            .groupby("badge")

            .size()

            .reset_index(name="employee_count")

            .sort_values(
                by="employee_count",
                ascending=False
            )

        )

        return summary

    ##################################################################
    # Save Results
    ##################################################################

    def b_save(self,df):

        df=df
       

        badge_file = self.config.badgeresults

        summary_file = self.config.badgesummary

        df.to_csv(
            badge_file,
            index=False
        )
        self.badge_summary(df).to_csv(
                            summary_file,
                            index=False
                        )
       

        print(f"Badge results saved to : {badge_file}")
        print(f"Badge summary saved to : {summary_file}")

        return df

    def calculate_level(self, df):

        print("##################################################################")
        print("# Calculate Level")
        print("##################################################################")

        level_numbers = []
        level_names = []
        next_level_points = []
        progress_percent = []

        # Sort levels by minimum points
        ordered_levels = sorted(
            self.params.gamification.levels.items(),
            key=lambda x: x[1]["min_points"]
        )

        for _, row in df.iterrows():

            points = row["points"]

            current_level = 1
            current_name = ordered_levels[0][1]["name"]
            current_min = ordered_levels[0][1]["min_points"]
            next_points = None

            # Find current level
            for index, (_, level) in enumerate(ordered_levels):

                if points >= level["min_points"]:

                    current_level = index + 1
                    current_name = level["name"]
                    current_min = level["min_points"]

                    if index < len(ordered_levels) - 1:
                        next_points = ordered_levels[index + 1][1]["min_points"]
                    else:
                        next_points = None

            # Highest level
            if next_points is None:

                progress = 100
                points_remaining = 0

            else:

                denominator = next_points - current_min

                if denominator <= 0:
                    progress = 100
                else:
                    progress = ((points - current_min) / denominator) * 100

                progress = max(0, min(progress, 100))
                points_remaining = max(0, next_points - points)

            level_numbers.append(current_level)
            level_names.append(current_name)
            next_level_points.append(points_remaining)
            progress_percent.append(round(progress, 2))

        df["level"] = level_numbers
        df["level_name"] = level_names
        df["points_to_next_level"] = next_level_points
        df["level_progress"] = progress_percent

        return df


    

    ####################################################################
    # Level Summary
    ####################################################################

    def level_summary(self,df):
        df=df

        summary = (

            df

            .groupby(["level", "level_name"])

            .size()

            .reset_index(name="employee_count")

            .sort_values(
                by="level",
                ascending=True
            )

        )

        return summary

    ####################################################################
    # Save Results
    ####################################################################

    def l_save(self,df):
        df=df
        levelresults =self.config.levelresults
        levelsummary = self.config.levelsummary

        df.to_csv(
            levelresults,
            index=False
        )

        self.level_summary(df).to_csv(
            levelsummary,
            index=False
        )

        print(f"Level results saved to : {levelresults}")
        print(f"Level summary saved to : {levelsummary}")

        return df


    ##############################################################
    # Overall Leaderboard
    ##############################################################

    def generate_leaderboard(self,df):
        df=df    
        print('##############################################################')
        print('# Overall Leaderboard')
        print('##############################################################')

        df = df.sort_values(

            by=[
                "total_points",
                "total_incidents"
            ],

            ascending=[
                False,
                False
            ]

        ).reset_index(drop=True)

        df["rank"] = df.index + 1

        return df

    ##############################################################
    # Assignment Group Leaderboard
    ##############################################################

    def assignment_group_leaderboard(self,df):
        df=df
        if "assignment_group" not in df.columns:

            return pd.DataFrame()

        group_df = self.df.copy()

        group_df["group_rank"] = (

            group_df

            .groupby("assignment_group")

            ["points"]

            .rank(

                method="dense",

                ascending=False

            )

        )

        return group_df

    ##############################################################
    # Top N Employees
    ##############################################################

    def top_performers(self,df):
        df=df
        top_n = self.params.gamification.leaderboard["top_n"]

        return df.head(top_n)

    ##############################################################
    # Bottom N Employees
    ##############################################################

    def bottom_performers(self,df):
        df=df

        top_n = self.params.gamification.leaderboard["top_n"]

        return df.tail(top_n)

    ##############################################################
    # Team Summary
    ##############################################################

    def team_summary(self,df):
        df=df

        if "assignment_group" not in df.columns:

            return pd.DataFrame()

        summary = (

            df

            .groupby("assignment_group")

            .agg(

                Employees=("assigned_to", "count"),

                Total_Points=("points", "sum"),

                Average_Level=("level", "mean"),

                Average_MTTR=("average_resolution", "mean")

            )

            .reset_index()

        )

        summary["Average_Level"] = summary[
            "Average_Level"
        ].round(2)

        summary["Average_MTTR"] = summary[
            "Average_MTTR"
        ].round(2)

        return summary

    ##############################################################
    # Save Results
    ##############################################################

    def lead_save(self, df):
        df=df
       
        leaderboardfile=self.config.leaderboardfile
        topfile=self.config.topfile
        bottomfile=self.config.bottomfile
        teamfile=self.config.teamfile
        groupfile=self.config.groupfile
 
        df.to_csv(
            leaderboardfile,
            index=False
        )
        self.top_performers(df).to_csv(
            topfile,
            index=False
        )

        self.bottom_performers(df).to_csv(
            bottomfile,
            index=False
        )

        self.team_summary(df).to_csv(
            teamfile,
            index=False
        )

        self.assignment_group_leaderboard(df).to_csv(
            groupfile,
            index=False
        )

        print(f"Leaderboard saved : {leaderboardfile}")

        print(f"Top performers saved : {topfile}")

        print(f"Bottom performers saved : {bottomfile}")

        print(f"Team summary saved : {teamfile}")

        print(f"Assignment Group Leaderboard : {groupfile}")
        return df
    ##############################################################
    # Achievement Calculation
    ##############################################################

    def calculate_achievements(self,df):
        df=df
        print('##############################################################')
        print('# Achievement Calculation')
        print('##############################################################')
        
        achievement_names = []
        achievement_count = []
        for _, row in df.iterrows():

            badges = []

            ##################################################
            # Incident Master
            ##################################################

            if row["total_incidents"] >= \
                    self.params.gamification.achievements["incident_master"]["incidents"]:

                badges.append("Incident Master")

            ##################################################
            # Point Master
            ##################################################

            if row["total_points"] >= \
                    self.params.gamification.achievements["point_master"]["points"]:

                badges.append("Point Master")

            ##################################################
            # SLA Champion
            ##################################################

            if "sla_percentage" in df.columns:

                if row["sla_percentage"] >= \
                        self.params.gamification.achievements["sla_champion"]["sla"]:

                    badges.append("SLA Champion")

            ##################################################
            # Speed Star
            ##################################################
        # Need to investigate 
            # if row["average_resolution"] <= \
            #         self.params.gamification.achievements["speed_star"]["mttr"]:

            #     badges.append("Speed Star")

            # ##################################################
            # # Elite Level
            # ##################################################

            # if row["level"] >= \
            #         self.params.gamification.achievements["elite_level"]["level"]:

            #     badges.append("Elite Level")

            ##################################################
            # Rank Achievement
            ##################################################

            if row["rank"] == \
                    self.params.gamification.achievements["top_rank"]["rank"]:

                badges.append("Top Performer")

            ##################################################

            if len(badges) == 0:

                badges.append("No Achievement")

            achievement_names.append(", ".join(badges))

            achievement_count.append(len(badges))

        df["achievements"] = achievement_names

        df["achievement_count"] = achievement_count

        return df

    ##############################################################
    # Achievement Summary
    ##############################################################

    def achievement_summary(self,df):
        df=df

        summary = (

            df

            [["assigned_to",
              "achievement_count",
              "achievements"]]

            .sort_values(

                by="achievement_count",
                ascending=False

            )

        )

        return summary

    ##############################################################
    # Save Results
    ##############################################################

    def ach_save(self, df):
        df=df
        achievementfile=self.config.achievementfile
        summaryfile=self.config.summaryfile
     
        df.to_csv(
            achievementfile,
            index=False
        )

        self.achievement_summary(df).to_csv(
            summaryfile,
            index=False
        )

        print(
            f"Achievement results saved : {achievementfile}"
        )

        print(
            f"Achievement summary saved : {summaryfile}"
        )

        return df

    ##############################################################
    # Generate Monthly Ranking
    ##############################################################

    def generate_monthly_ranking(self):

        if "resolved_at" not in self.df.columns:

            raise Exception(
                "resolved_at column is missing."
            )

        self.df["resolved_at"] = pd.to_datetime(
            self.df["resolved_at"]
        )

        self.df["month"] = self.df[
            "resolved_at"
        ].dt.to_period("M").astype(str)

        monthly = (

            self.df

            .groupby(

                [
                    "month",
                    "assigned_to"
                ]

            )

            .agg(

                total_points=("points", "sum"),

                total_incidents=("number", "count"),

                average_resolution=(

                    "resolution_hours",

                    "mean"

                )

            )

            .reset_index()

        )

        monthly["average_resolution"] = (

            monthly["average_resolution"]

            .round(2)

        )

        monthly = monthly.sort_values(

            by=[

                "month",

                "total_points",

                "total_incidents"

            ],

            ascending=[

                True,

                False,

                False

            ]

        )

        monthly["monthly_rank"] = (

            monthly

            .groupby("month")

            .cumcount()

            + 1

        )

        self.monthly_df = monthly

        return self.monthly_df

    ##############################################################
    # Top N Per Month
    ##############################################################

    def top_performers(self):

        top_n = self.params["top_n"]

        top = (

            self.monthly_df

            .groupby("month")

            .head(top_n)

        )

        return top

    ##############################################################
    # Monthly Summary
    ##############################################################

    def monthly_summary(selfdf,df):
        monthly_df=df
        summary = (

            monthly_df

            .groupby("month")

            .agg(

                Employees=(

                    "assigned_to",

                    "count"

                ),

                TotalPoints=(

                    "total_points",

                    "sum"

                ),

                Incidents=(

                    "total_incidents",

                    "sum"

                )

            )

            .reset_index()

        )

        return summary

    ##############################################################
    # Save Results
    ##############################################################

    def save(self,df):
        monthly_df=df
        monthlyranking=self.config.monthlyranking
        monthlytopfile =self.config.monthlytopfile
        monthlysummaryfile=self.config.monthlysummaryfile

        monthly_df.to_csv(
            monthlyranking,
            index=False
        )

        self.top_performers(monthly_df).to_csv(
            monthlytopfile,
            index=False
        )

        self.monthly_summary(monthly_df).to_csv(
            monthlysummaryfile,
            index=False
        )

        print(
            f"Monthly Ranking saved : {monthlyranking}"
        )

        print(
            f"Top Performers saved : {monthlytopfile}"
        )
        print(
            f"Monthly Summary saved : {monthlysummaryfile}"
        )
    
    

In [19]:
config = ConfigurationManager()

gamification_config = config.get_datagamificationconfig()

gamification = Gamification(gamification_config)   
df=gamification.calculate_points()
df_f,df_analyst_summary=gamification.p_save(df)
df_b=gamification.assign_badges(df_f)
df=gamification.b_save(df_b)
df_l=gamification.calculate_level(df)
df=gamification.l_save(df_l)
df_lead=gamification.generate_leaderboard(df_analyst_summary)
df=gamification.lead_save(df_lead)
df_ach=gamification.calculate_achievements(df_lead)
gamification.ach_save(df_ach)




Config\Config.yaml
path config yaml Config\Config.yaml
Reading YAML : Config\Config.yaml
[2026-08-02 15:21:54,694: INFO: Common: YAML file loaded successfully : Config\Config.yaml]
Reading YAML : params.yaml
[2026-08-02 15:21:54,711: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-08-02 15:21:54,715: INFO: Common: Created directory : Artifacts/DataGamification]
Reading YAML : params.yaml
[2026-08-02 15:21:54,739: INFO: Common: YAML file loaded successfully : params.yaml]
##################################################################
 # Point calculation
##################################################################
Incident points saved to : Artifacts/DataGamification/IncidentPoints.csv
Analyst summary saved to : Artifacts/DataGamification/AnalystPoints.csv
##################################################################
 #  Badge Assignment
##################################################################
Badge results saved to : Artifacts/DataGamification/

,assigned_to,total_incidents,total_points,sla_met,average_resolution,rank,achievements,achievement_count
0,Venkat,15,415.0,4,462.65,1,Top Performer,1
1,Chaitanya,10,285.0,2,882.00,2,No Achievement,1
2,Raju,15,165.0,0,798.84,3,No Achievement,1
3,Vamsi,15,165.0,1,792.80,4,No Achievement,1
4,Ganesh,14,140.0,1,826.67,5,No Achievement,1
